In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print("successfully imported libraries")
print("pandas version:",pd.__version__)
print("requests version",requests.__version__)


successfully imported libraries
pandas version: 2.2.2
requests version 2.32.4


In [ ]:
df=pd.read_csv("messy_sales_data.csv")
print("number of columns=",df.shape[1])
print("number of rows=",df.shape[0])
print("columns:",df.columns.to_list)
print(df.head())

number of columns= 9
number of rows= 30
columns: <bound method IndexOpsMixin.tolist of Index(['order_id', 'customer_name', 'product', 'category', 'quantity',
       'unit_price', 'order_date', 'city', 'sales_rep'],
      dtype='object')>
   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
4  2024-01-05     Mumbai  Anil Sharma  


In [ ]:
print("="*40)
print("Data quality diagnosis report")
print("="*40)
print("\n[1] Missing values per column")
print(df.isnull().sum())

print(f"\n[2] Duplicate rows:{df.duplicated().sum()}")
print("\n[3] data types:")
print(df.dtypes)
print("\n[4] UNIQUE CATEGORIES:",df['category'].unique())
print("\n[5] Sample customer names:",df['customer_name'].dropna().unique()[:8])
print("\n[6] ")

Data quality diagnosis report

[1] Missing values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] Duplicate rows:0

[3] data types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]

[5] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']

[6] 


In [ ]:
df1=df.copy()
print(f"working copy created:{df.shape}")
print("df1 is untouched- we can always reset by running df11=df1.copy()")

working copy created:(30, 9)
df1 is untouched- we can always reset by running df11=df1.copy()


In [ ]:
print("Before fixing nulls:",df1.isnull().sum().sum(), "total missing values")
df1['customer_name'].fillna('Unknown customer',inplace=True)
median_qty=df1['quantity'].median()
df1['quantity'].fillna(median_qty,inplace=True)
print(f"Filled missing quantity with median:{median_qty}")
df1['category'].fillna('Uncategorised',inplace=True)
print("After fixing nulls:",df1.isnull().sum().sum(), "total missing values")

Before fixing nulls: 7 total missing values
Filled missing quantity with median:2.0
After fixing nulls: 1 total missing values


In [ ]:
print(f"before dedupliclation:{len(df1)}rows")
print(f"Duplicate rows:{df1.duplicated().sum()}")
print("Duplicated rows")
print(df1[df1.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
df1.drop_duplicates(inplace=True)
print(f"\n After deduplication: {len(df1)} rows")
print(f"\n Rows removed:{len(df)-len(df1)}")


before dedupliclation:30rows
Duplicate rows:0
Duplicated rows
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

 After deduplication: 30 rows

 Rows removed:0


In [ ]:
print('Sample dates before parsing:')
print(df1['order_date'].head(8).tolist())
df1['order_date'] = pd.to_datetime(
df1['order_date'],
dayfirst=False,
errors='coerce'
)
net_count=df1['order_date'].isnull().sum()
print(f"\nUnparsable dates (NaT):{net_count}")
df1['year']=df1['order_date'].dt.year
df1['month']=df1['order_date'].dt.month
df1['month_name']=df1['order_date'].dt.strftime('%B')

print("\nSample dates after parsing:")
print(df1[['order_date','year','month','month_name']].head(5))



Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparsable dates (NaT):2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [ ]:

print("Before standardization",df1['customer_name'].unique()[:6])
df1['customer_name']=(df1['customer_name'].str.strip().str.title())
print("after standardization:",df1['customer_name'].unique()[:6])
print(f"\nBefore: keyboard rows with Electronics category:")
wrong_mask=(df1['product']=='keyboard')&(df1['category']=='Electronics')
df1.loc[wrong_mask,'category']='Accessories'
print("After fix : unique categories:",df1['category'].unique())


Before standardization ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
after standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before: keyboard rows with Electronics category:
After fix : unique categories: ['Electronics' 'Accessories' 'Uncategorised']


In [ ]:

df1['unit_price']=pd.to_numeric(df1['unit_price'],errors='coerce')
df1['revenue']=df1['quantity']*df1['unit_price']
print("Remove column created:")
print(df1[['customer_name','product','quantity','unit_price','revenue']].head(5))
print(f"\nTotal revenue across all orders:₹ {df1['revenue'].sum():,.0f}")


Remove column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

Total revenue across all orders:₹ 818,000


In [ ]:

print(df1.isnull().sum().sum())
print("="* 55)
print('POST CLEANING VALIDATION REPORT')
print("="*55)
print("Original row:",len(df))
print("Cleaned rows:",len(df1))
print("Rows removed:",len(df)-len(df1))
print("Missing values:",df1.isnull().sum().sum())
print("Duplicates:",df1.duplicated().sum())
print("Date nulls:",df1['order_date'].isnull().sum())
print("Revenue NaN:",df1['revenue'].isnull().sum())
print("caregories:",sorted(df1['category'].unique()))
print("="*55)
all_clean=(
    df1.isnull().sum().sum()==0 and
    df1.duplicated().sum()==0

)
print(f"data is clean:{all_clean}")



9
POST CLEANING VALIDATION REPORT
Original row: 30
Cleaned rows: 30
Rows removed: 0
Missing values: 9
Duplicates: 0
Date nulls: 2
Revenue NaN: 0
caregories: ['Accessories', 'Electronics', 'Uncategorised']
data is clean:False


In [ ]:
print()

    order_id     customer_name     product       category  quantity  \
0       1001      Ramesh Kumar      Laptop    Electronics       2.0   
1       1002        Priya Nair         NaN    Electronics       1.0   
2       1003        Amit Verma    Keyboard    Accessories       3.0   
3       1004      Sunita Patel     Monitor    Electronics       2.0   
4       1005      Ramesh Kumar      Laptop    Electronics       2.0   
5       1006       Kiran Mehta       Mouse    Accessories      10.0   
6       1007      Deepak Singh  Headphones    Electronics       2.0   
7       1008  Unknown Customer      Webcam    Accessories       1.0   
8       1009        Ananya Das      Laptop    Electronics       1.0   
9       1010       Vikram Iyer    Keyboard    Accessories       5.0   
10      1011       Pooja Gupta     Monitor    Electronics       2.0   
11      1012        Suresh Rao     USB Hub    Accessories       8.0   
12      1013       Meera Joshi      Laptop    Electronics       2.0   
13    

In [ ]:
product_rev = (
    df1.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue', ascending=False)
)
print("Revenue by Product:")
print(product_rev.to_string(index=False))
category_summary = (
    df1.groupby('category')
    .agg(
        total_revenue=('revenue', 'sum'),
        avg_order_value=('revenue', 'mean'),
        num_orders=('order_id', 'count'),
        unique_products=('product', 'nunique')
    )
    .round(2)
    .reset_index()
)
print("\nCategory Summary:")
print(category_summary.to_string(index=False))

Revenue by Product:
   product  revenue
    Laptop 540000.0
   Monitor 154000.0
Headphones  28000.0
     Mouse  20800.0
  Keyboard  20400.0
    Webcam  20000.0
   USB Hub  19800.0

Category Summary:
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories        76200.0          5861.54          13                4
  Electronics       697800.0         43612.50          16                4
Uncategorised        44000.0         44000.00           1                1


In [ ]:
df1.to_csv('Clean_sales_data.csv', index=False)
print('Cleaned data saved to : clean_sales_data.csv')
print(f'Final dataset: {df1.shape[0]} rows * {df1.shape[1]} columns')
print('\nETL Pipeline for sales Data: COMPLETE')
print('  EXTRACT -> messy_sales_data.csv loaded')
print('  TRANSFORM -> nulls fixed, dupes removed, data types standardized')
print('  LOAD -> Clean_sales_data.csv saved')

Cleaned data saved to : clean_sales_data.csv
Final dataset: 30 rows * 13 columns

ETL Pipeline for sales Data: COMPLETE
  EXTRACT -> messy_sales_data.csv loaded
  TRANSFORM -> nulls fixed, dupes removed, data types standardized
  LOAD -> Clean_sales_data.csv saved


In [1]:
API_KEY="b2aef8ac9375d7e143cfcadfb7bca30e"
BASE_URL="https://api.openweathermap.org/data/2.5/weather"
CITIES=['Mumbai','Delhi','Bangalore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']
print(f"API configured for {len(CITIES)} cities")
print(f"Cities: {CITIES}")
print("\n IMPORTANT:Replace your api key here with your actual key before running")

API configured for 8 cities
Cities: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

 IMPORTANT:Replace your api key here with your actual key before running


In [6]:
import requests;
def fetch_weather(city,API_KEY):
  params={
  'q': city,
  'appid': API_KEY,
  'units': 'metric'
  }
  try:
    response=requests.get(BASE_URL,params=params,timeout=10)
    if response.status_code==200:
      data=response.json()
      return{
          'city':city,
          'temperature': round(data['main']['temp'],1),
          'feels_like': round(data['main']['feels_like'],1),
          'humidity': data['main']['humidity'],
          'pressure': data['main']['pressure'],
          'wind_speed': data['wind']['speed'],
          'condition': data['weather'][0]['description'].title(),
          'visibility':data.get('visibility',0)

      }
    else:
      print(f"ERROR{response.status_code} for {city}: {response.json().get("message","unknown errror")}")
      return None
  except requests.exceptions.ConnectionError:
    print(f" CONNECTION ERROR FOR {city} -Check internet connection")
    return None
  except request.exceptions.Timeout:
    print(f"TIMEOUT for {city}-API did not respond in 10 seconds")
    return None
print("Calliing weather api...")
weather_records=[]
for city in CITIES:
  print(f" Fteching :{city}...",end="")
  record=fetch_weather(city,API_KEY)
  if record:
    weather_records.append(record)
    print(f"{record['temperature']}°C, {record["condition"]}")
  else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')



Calliing weather api...
 Fteching :Mumbai...32.0°C, Haze
 Fteching :Delhi...32.0°C, Thunderstorm With Light Rain
 Fteching :Bangalore...28.6°C, Scattered Clouds
 Fteching :Chennai...32.3°C, Scattered Clouds
 Fteching :Hyderabad...36.2°C, Scattered Clouds
 Fteching :Kolkata...27.0°C, Broken Clouds
 Fteching :Pune...31.4°C, Clear Sky
 Fteching :Jaipur...40.6°C, Haze

Successfully fetched: 8/8 cities


In [ ]:
if len(weather_records) == 0:
    print('Using fallback weather data (API not available)')
    weather_records = [
        {'city':'Mumbai',    'temperature':32.5,'feels_like':36.0,'humidity':78,'pressure':1009,'wind_speed':5.2,'condition':'Partly Cloudy','visibility':8},
        {'city':'Delhi',     'temperature':38.2,'feels_like':41.0,'humidity':35,'pressure':1002,'wind_speed':3.8,'condition':'Clear Sky',    'visibility':10},
        {'city':'Bangalore', 'temperature':26.1,'feels_like':27.0,'humidity':62,'pressure':1016,'wind_speed':2.5,'condition':'Overcast',     'visibility':7},
        {'city':'Chennai',   'temperature':34.8,'feels_like':39.0,'humidity':72,'pressure':1008,'wind_speed':6.1,'condition':'Hazy',         'visibility':5},
        {'city':'Hyderabad', 'temperature':35.4,'feels_like':38.5,'humidity':45,'pressure':1005,'wind_speed':4.2,'condition':'Clear Sky',    'visibility':10},
        {'city':'Kolkata',   'temperature':33.7,'feels_like':37.8,'humidity':80,'pressure':1007,'wind_speed':4.8,'condition':'Humid',        'visibility':6},
        {'city':'Pune',      'temperature':29.3,'feels_like':31.0,'humidity':55,'pressure':1014,'wind_speed':3.1,'condition':'Partly Cloudy','visibility':9},
        {'city':'Jaipur',    'temperature':40.1,'feels_like':43.0,'humidity':22,'pressure':998, 'wind_speed':5.5,'condition':'Sunny',        'visibility':12},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live API data for {len(weather_records)} cities')

In [8]:
import pandas as pd
weather_df = pd.DataFrame(weather_records)
# pd.DataFrame() converts a list of dictionaries into a DataFrame
# Each dictionary becomes one row
# Each dictionary key becomes a column name
# This is the standard pattern for API → DataFrame conversion

print('Weather DataFrame created:')
print(weather_df.to_string(index=False))
print(f'\nShape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')
print(f'\nData types:')
print(weather_df.dtypes)

Weather DataFrame created:
     city  temperature  feels_like  humidity  pressure  wind_speed                    condition  visibility
   Mumbai         32.0        39.0        66      1008        4.63                         Haze        4000
    Delhi         32.0        36.6        58       998        6.17 Thunderstorm With Light Rain        3500
Bangalore         28.6        30.9        64      1011        1.03             Scattered Clouds        6000
  Chennai         32.3        39.3        74      1006        5.14             Scattered Clouds        6000
Hyderabad         36.2        37.6        34      1005        3.60             Scattered Clouds        6000
  Kolkata         27.0        30.6        89      1006        7.72                Broken Clouds        3200
     Pune         31.4        32.4        46      1009        6.25                    Clear Sky       10000
   Jaipur         40.6        40.6        21      1000        6.69                         Haze        4500



In [9]:
print('=' * 50)
print('  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES')
print('=' * 50)

# Hottest and coldest city
hottest = weather_df.loc[weather_df['temperature'].idxmax()]
coldest = weather_df.loc[weather_df['temperature'].idxmin()]
# .idxmax() returns the INDEX of the maximum value
# .idxmin() returns the INDEX of the minimum value
# df.loc[index] returns the full row at that index

print(f"\nHottest city : {hottest['city']} at {hottest['temperature']}°C")
print(f"Coldest city : {coldest['city']} at {coldest['temperature']}°C")
print(f"Most humid   : {weather_df.loc[weather_df['humidity'].idxmax()]['city']} "
      f"({weather_df['humidity'].max()}%)")

# Summary statistics
print(f"\nAverage temperature : {weather_df['temperature'].mean():.1f}°C")
print(f"Average humidity    : {weather_df['humidity'].mean():.1f}%")
print(f"Average wind speed  : {weather_df['wind_speed'].mean():.1f} m/s")

# Rankings
print('\nCities ranked by temperature (hottest first):')
ranked = weather_df[['city','temperature','humidity']].sort_values('temperature', ascending=False)
print(ranked.to_string(index=False))

  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES

Hottest city : Jaipur at 40.6°C
Coldest city : Kolkata at 27.0°C
Most humid   : Kolkata (89%)

Average temperature : 32.5°C
Average humidity    : 56.5%
Average wind speed  : 5.2 m/s

Cities ranked by temperature (hottest first):
     city  temperature  humidity
   Jaipur         40.6        21
Hyderabad         36.2        34
  Chennai         32.3        74
   Mumbai         32.0        66
    Delhi         32.0        58
     Pune         31.4        46
Bangalore         28.6        64
  Kolkata         27.0        89


In [10]:
weather_df.to_csv('weather_data.csv', index=False)
# Saves the cleaned, structured weather DataFrame to a CSV file
# This file can be loaded into SQLite (Day 2 skills) or used in ML (Day 5)

print('Weather data saved to: weather_data.csv')
print('\nWeather ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called for 8 cities')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print('  LOAD      → weather_data.csv saved')

Weather data saved to: weather_data.csv

Weather ETL Pipeline: COMPLETE
  EXTRACT   → OpenWeatherMap API called for 8 cities
  TRANSFORM → JSON parsed, DataFrame built, units converted
  LOAD      → weather_data.csv saved


1. What are the three stagesof ETL? Describe each stage using an example from today's sales dtaset
2. A Data frame has 500 rows. After calling df.dropna() , it has 412 rows.what does this will  you?

3. write a code to remove the duplicate from df where "same row" means same customer_name and same product

4. whatis the difference between fillna(0) and fillna(df['col'].meadian())? when do you prefer each?

5. write on python code to call the weather API for delhi and print the temperature in celsius?

6. What does response.status_code==200 mean? what should you do when the code is 401?

what are the three stages of ult?

Answer:

U — Upload (or Extract)
Data is collected from different sources such as databases, APIs, files, applications, or sensors.
Example: Collecting sales data from an e-commerce website.

L — Load
The raw data is loaded directly into a data warehouse or data lake without heavy preprocessing.
Example: Loading raw CSV files into Snowflake, BigQuery, or Hadoop.

T — Transform
After loading, the data is cleaned, filtered, joined, aggregated, and converted into useful formats inside the target system itself.
Example: Removing null values, calculating totals, or creating reports using SQL.

A Data frame has 500 rows. After calling df.dropna() , it has 412 rows.what does this will you?

Answer:

The original DataFrame contained 500 rows. After applying df.dropna(), the number of rows reduced to 412. This indicates that 88 rows were removed because they contained missing (NaN) values.

what is the difference between fillna(0) and fillna(df['col'].meadian())? when do you prefer each?

Answer:
fillna(0) replaces all missing values with 0, whereas fillna(df['col'].median()) replaces missing values with the median value of that specific column.

Use fillna(0) when missing values logically represent zero or absence, such as number of items sold, marks not scored, or count-based data. It is simple and useful when zero has meaningful importance in the dataset.

Use fillna(df['col'].median()) when working with numerical data where missing values should be replaced with a typical value from the dataset. The median is preferred especially when the data contains outliers because it is less affected by extreme values compared to the mean.

What does response.status_code==200 mean? what should you do when the code is 401?
Answer:

response.status_code == 200 means that the HTTP request was successful and the server returned the requested data correctly.

A status code 401 means Unauthorized. It indicates that authentication failed or the user does not have valid access credentials.

In [12]:
#5. write on python code to call the weather API for delhi and print the temperature in celsius?
import requests;
API_KEY="b2aef8ac9375d7e143cfcadfb7bca30e"
BASE_URL="https://api.openweathermap.org/data/2.5/weather"
CITIES=['Delhi']

def fetch_weather(city,API_KEY):
  params={
  'q': city,
  'appid': API_KEY,
  'units': 'metric'
  }
  try:
    response=requests.get(BASE_URL,params=params,timeout=10)
    if response.status_code==200:
      data=response.json()
      return{
          'city':city,
          'temperature': round(data['main']['temp'],1),
          'feels_like': round(data['main']['feels_like'],1),
          'humidity': data['main']['humidity'],
          'pressure': data['main']['pressure'],
          'wind_speed': data['wind']['speed'],
          'condition': data['weather'][0]['description'].title(),
          'visibility':data.get('visibility',0)

      }
    else:
      print(f"ERROR{response.status_code} for {city}: {response.json().get("message","unknown errror")}")
      return None
  except requests.exceptions.ConnectionError:
    print(f" CONNECTION ERROR FOR {city} -Check internet connection")
    return None
  except requests.exceptions.Timeout:
    print(f"TIMEOUT for {city}-API did not respond in 10 seconds")
    return None
print("Calliing weather api...")
weather_records=[]
for city in CITIES:
  print(f" Fteching :{city}...",end="")
  record=fetch_weather(city,API_KEY)
  if record:
    weather_records.append(record)
    print(f"{record['temperature']}°C, {record["condition"]}")
  else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')




Calliing weather api...
 Fteching :Delhi...32.0°C, Thunderstorm With Light Rain

Successfully fetched: 1/1 cities
